In [ ]:
!pip install transformers datasets gradio

In [ ]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    pipeline,
)
from datasets import Dataset as HFDataset  # Renaming to avoid conflict
import numpy as np
from tqdm import tqdm

# For the Gradio interface (optional)
try:
    import gradio as gr
except ImportError:
    print("Gradio library not found. Install it using: pip install gradio")
    gr = None


In [ ]:
# --- 1. Dataset Preparation ---

class CustomTextDataset(Dataset):
    def __init__(self, file_paths, tokenizer, max_length):
        self.tokenizer = tokenizer
        self.texts = []
        for category, file_path in file_paths.items():
            with open(file_path, 'r', encoding='utf-8') as f:
                for line in f:
                    text = line.strip()
                    if text:
                        self.texts.append(f"[{category.upper()}] {text}")
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': encoding['input_ids'].squeeze()  # Labels are the input IDs for causal language modeling
        }

def load_and_prepare_dataset(file_paths, model_name, max_length=512):
    """Loads text data from files, prefixes with category labels, and tokenizes it."""
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # Add special tokens for the categories if they are not already present
    special_tokens_dict = {'additional_special_tokens': ["[NEWS]", "[STORY]", "[JOKE]"]}
    tokenizer.add_special_tokens(special_tokens_dict)

    dataset = CustomTextDataset(file_paths, tokenizer, max_length)
    return dataset, tokenizer

In [ ]:
# --- 2. Model Setup ---

def setup_model(model_name, tokenizer):
    """Loads a pre-trained causal language model and resizes the embedding layer."""
    model = AutoModelForCausalLM.from_pretrained(model_name)

    # Important: Resize the embedding layer to account for the new special tokens
    model.resize_token_embeddings(len(tokenizer))
    return model

In [ ]:
# --- 3. Training Loop ---

def train_model(model, tokenizer, train_dataset, output_dir="output", batch_size=8, learning_rate=5e-5, epochs=3):
    """Fine-tunes the pre-trained model on the custom dataset."""
    # Remove 'evaluation_strategy' as it's causing the error
    training_args = TrainingArguments(
        output_dir=output_dir,
        overwrite_output_dir=True,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        save_steps=1000,
        save_total_limit=2,
        learning_rate=learning_rate,
        logging_dir='./logs',
        logging_steps=100,
        # evaluation_strategy="epoch",  # Remove this line
        per_device_eval_batch_size=batch_size,
        dataloader_num_workers=2,
        remove_unused_columns=False, # Keep 'labels'
    )

In [ ]:
# --- 4. Generation ---

def generate_text(model, tokenizer, prompt, max_length=150, temperature=0.7, num_return_sequences=1):
    """Generates text based on a given prompt."""
    generator = pipeline('text-generation', model=model, tokenizer=tokenizer, device=model.device)
    generated_texts = generator(
        prompt,
        max_length=max_length,
        num_return_sequences=num_return_sequences,
        temperature=temperature,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id, # Important for some models
    )
    return [output['generated_text'] for output in generated_texts]


In [ ]:
# --- 5. Deployment (Gradio) ---

def create_gradio_interface(model, tokenizer):
    """Creates a simple Gradio interface for interacting with the model."""
    def predict(prompt, max_length, temperature):
        generated_texts = generate_text(model, tokenizer, prompt, int(max_length), temperature)
        return "\n\n".join(generated_texts)

    if gr:
        iface = gr.Interface(
            fn=predict,
            inputs=[
                gr.Textbox(label="Enter Prompt (e.g., '[NEWS] Latest tech trends')", lines=2),
                gr.Number(value=150, minimum=50, maximum=500, step=25, label="Max Length"),
                gr.Number(value=0.7, minimum=0.1, maximum=1.0, step=0.1, label="Temperature"),
            ],
            outputs=gr.Textbox(label="Generated Text", lines=5),
            title="Custom Text Generator",
            description="Generate news, stories, or jokes by starting your prompt with '[NEWS]', '[STORY]', or '[JOKE]'.",
        )
        iface.launch(share=True)
    else:
        print("Gradio not installed, skipping interface creation.")


In [ ]:
# --- Main Execution ---

if __name__ == "__main__":
    # Define file paths for your datasets
    file_paths = {
        "news": "news.txt",
        "story": "story.txt",
        "joke": "jokes.txt",
    }

    # Create dummy data files if they don't exist for demonstration
    for category, file_path in file_paths.items():
        if not os.path.exists(file_path):
            with open(file_path, 'w', encoding='utf-8') as f:
                f.write(f"This is a sample {category} article.\n")
                f.write(f"Another example of a short {category}.\n")

    # Choose a pre-trained model
    model_name = "gpt2"  # You can also try "distilgpt2"

    # Prepare the dataset and tokenizer
    train_dataset, tokenizer = load_and_prepare_dataset(file_paths, model_name)

    # Setup the model
    model = setup_model(model_name, tokenizer)
    model.to("cuda" if torch.cuda.is_available() else "cpu") # Move model to GPU if available

    # Train the model
    trainer = train_model(model, tokenizer, train_dataset, epochs=10) # Reduced epochs for demonstration

    # Save the trained model
    model.save_pretrained("custom_text_generator")
    tokenizer.save_pretrained("custom_text_generator")
    print("Trained model and tokenizer saved to 'custom_text_generator' directory.")

    # Load the trained model for generation
    trained_model = AutoModelForCausalLM.from_pretrained("custom_text_generator").to("cuda" if torch.cuda.is_available() else "cpu")
    trained_tokenizer = AutoTokenizer.from_pretrained("custom_text_generator")

    # Example Generation
    news_prompt = "[NEWS] Breaking news: AI achieves"
    generated_news = generate_text(trained_model, trained_tokenizer, news_prompt)
    print(f"\nGenerated News: {generated_news[0]}")

    story_prompt = "[STORY] Once upon a time, in a land far away"
    generated_story = generate_text(trained_model, trained_tokenizer, story_prompt)
    print(f"Generated Story: {generated_story[0]}")

    joke_prompt = "[JOKE] Why don't scientists trust atoms?"
    generated_joke = generate_text(trained_model, trained_tokenizer, joke_prompt)
    print(f"Generated Joke: {generated_joke[0]}")

    # Create Gradio interface (optional)
    create_gradio_interface(trained_model, trained_tokenizer)


Trained model and tokenizer saved to 'custom_text_generator' directory.
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://391f6e64b2488a84f6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
!pip install transformers datasets gradio

import os
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    pipeline,
)
from datasets import Dataset as HFDataset
import numpy as np
from tqdm import tqdm

# --- 1. Dataset Preparation ---

class CustomTextDataset(Dataset):
    def __init__(self, file_paths, tokenizer, max_length):
        self.tokenizer = tokenizer
        self.texts = []
        for category, file_path in file_paths.items():
            with open(file_path, 'r', encoding='utf-8') as f:
                for line in f:
                    text = line.strip()
                    if text:
                        self.texts.append(f"[{category.upper()}] {text}")
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': encoding['input_ids'].squeeze()  # Labels are the input IDs for causal language modeling
        }

def load_and_prepare_dataset(file_paths, model_name, max_length=512):
    """Loads text data from files, prefixes with category labels, and tokenizes it."""
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # Add special tokens for the categories if they are not already present
    special_tokens_dict = {'additional_special_tokens': ["[NEWS]", "[STORY]", "[JOKE]"]}
    tokenizer.add_special_tokens(special_tokens_dict)

    dataset = CustomTextDataset(file_paths, tokenizer, max_length)
    return dataset, tokenizer

# --- 2. Model Setup ---

def setup_model(model_name, tokenizer):
    """Loads a pre-trained causal language model and resizes the embedding layer."""
    model = AutoModelForCausalLM.from_pretrained(model_name)

    # Important: Resize the embedding layer to account for the new special tokens
    model.resize_token_embeddings(len(tokenizer))
    return model

# --- 3. Training Loop ---

def train_model(model, tokenizer, train_dataset, output_dir="output", batch_size=8, learning_rate=5e-5, epochs=3):
    """Fine-tunes the pre-trained model on the custom dataset."""
    # Remove 'evaluation_strategy' as it's causing the error
    training_args = TrainingArguments(
        output_dir=output_dir,
        overwrite_output_dir=True,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        save_steps=1000,
        save_total_limit=2,
        learning_rate=learning_rate,
        logging_dir='./logs',
        logging_steps=100,
        # evaluation_strategy="epoch",  # Remove this line
        per_device_eval_batch_size=batch_size,
        dataloader_num_workers=2,
        remove_unused_columns=False, # Keep 'labels'
    )
    # WARNING: Missing Trainer initialization and training execution

# --- 4. Generation ---

def generate_text(model, tokenizer, prompt, max_length=150, temperature=0.7, num_return_sequences=1):
    """Generates text based on a given prompt."""
    generator = pipeline('text-generation', model=model, tokenizer=tokenizer, device=model.device)
    generated_texts = generator(
        prompt,
        max_length=max_length,
        num_return_sequences=num_return_sequences,
        temperature=temperature,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id, # Important for some models
    )
    return [output['generated_text'] for output in generated_texts]

# --- 5. Deployment (Gradio) ---

def create_gradio_interface(model, tokenizer):
    """Creates a simple Gradio interface for interacting with the model."""
    def predict(prompt, max_length, temperature):
        generated_texts = generate_text(model, tokenizer, prompt, int(max_length), temperature)
        return "\n\n".join(generated_texts)

    if gr:
        iface = gr.Interface(
            fn=predict,
            inputs=[
                gr.Textbox(label="Enter Prompt (e.g., '[NEWS] Latest tech trends')", lines=2),
                gr.Number(value=150, minimum=50, maximum=500, step=25, label="Max Length"),
                gr.Number(value=0.7, minimum=0.1, maximum=1.0, step=0.1, label="Temperature"),
            ],
            outputs=gr.Textbox(label="Generated Text", lines=5),
            title="Custom Text Generator",
            description="Generate news, stories, or jokes by starting your prompt with '[NEWS]', '[STORY]', or '[JOKE]'.",
        )
        iface.launch(share=True, debug = True)
    else:
        print("Gradio not installed, skipping interface creation.")

# --- Main Execution ---

if __name__ == "__main__":
    # Define file paths for your datasets
    file_paths = {
        "news": "news.txt",
        "story": "story.txt",
        "joke": "jokes.txt",
    }

    # Create dummy data files if they don't exist for demonstration
    for category, file_path in file_paths.items():
        if not os.path.exists(file_path):
            with open(file_path, 'w', encoding='utf-8') as f:
                f.write(f"This is a sample {category} article.\n")
                f.write(f"Another example of a short {category}.\n")

    # Choose a pre-trained model
    model_name = "gpt2"  # You can also try "distilgpt2"

    # Prepare the dataset and tokenizer
    train_dataset, tokenizer = load_and_prepare_dataset(file_paths, model_name)

    # Setup the model
    model = setup_model(model_name, tokenizer)
    model.to("cuda" if torch.cuda.is_available() else "cpu") # Move model to GPU if available

    # Train the model
    trainer = train_model(model, tokenizer, train_dataset, epochs=10) # Reduced epochs for demonstration

    # Save the trained model
    model.save_pretrained("custom_text_generator")
    tokenizer.save_pretrained("custom_text_generator")
    print("Trained model and tokenizer saved to 'custom_text_generator' directory.")

    # Load the trained model for generation
    trained_model = AutoModelForCausalLM.from_pretrained("custom_text_generator").to("cuda" if torch.cuda.is_available() else "cpu")
    trained_tokenizer = AutoTokenizer.from_pretrained("custom_text_generator")

    # Example Generation
    news_prompt = "[NEWS] Breaking news: AI achieves"
    generated_news = generate_text(trained_model, trained_tokenizer, news_prompt)
    print(f"\nGenerated News: {generated_news[0]}")

    story_prompt = "[STORY] Once upon a time, in a land far away"
    generated_story = generate_text(trained_model, trained_tokenizer, story_prompt)
    print(f"Generated Story: {generated_story[0]}")

    joke_prompt = "[JOKE] Why don't scientists trust atoms?"
    generated_joke = generate_text(trained_model, trained_tokenizer, joke_prompt)
    print(f"Generated Joke: {generated_joke[0]}")

    # Create Gradio interface (optional)
    create_gradio_interface(trained_model, trained_tokenizer)

ERROR: Operation cancelled by user
Trained model and tokenizer saved to 'custom_text_generator' directory.


Device set to use cuda:0
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Device set to use cuda:0



Generated News: [NEWS] Breaking news: AI achieves first step on artificial intelligence

The new study, which says it will be published in the Journal of Computational Neuroscience, suggests that our brains may be capable of helping us solve problems that we don't even know how to solve.

Researchers used a computer program to learn how to recognize a language. The program then programmed the computer to think that the program was looking for a word, and the program then said yes.

These results show that the program did know enough to think that the word was real. And they suggest that the program may have evolved to use the correct information rather than being forced to think by a human.

"This may be the first study of its kind that has been


Device set to use cuda:0


Generated Story: [STORY] Once upon a time, in a land far away from the bustling city of Phoenicia, the king of the people had a wife. This was the daughter of the king's friend, the king's nephew.

The king's niece, the daughter of the king's friend, was a woman. She was the most beautiful of all the members of the family, which was what the king's niece was. She was the daughter of the king's nephew, the king's nephew's daughter. That was one of the most beautiful people on Earth.

The king's niece had already married the queen of the people. The king's niece had already taken care of her sister's mother. The king's niece had already died and
Generated Joke: [JOKE] Why don't scientists trust atoms? The fact is, they don't. And the evidence was pretty clear: there was no solid evidence that atoms were formed from the same chemical reactions. A lot of that stuff is still going on. (The original paper from 2009 was published in the journal Nature.)

But now, thanks to a small group of re

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://5c56fb60b79d6d976d.gradio.live
Killing tunnel 127.0.0.1:7861 <> https://933fe1e05b30fa16b7.gradio.live
